# Entrenamiento del clasificador de estados de tráfico

Workflow de entrenamiento batch de VAAET ML 4.0.0. Consume telemetría adquirida bajo demanda, genera las 19 features canónicas, entrena el MLP y exporta el bundle portable de cuatro archivos.

In [ ]:
# Environment setup — run once per Colab runtime
import importlib.metadata
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/zgfnicolas/vaaet.git"
REPO_DIR = Path("/content/vaaet")

if IN_COLAB:
    if (REPO_DIR / ".git").is_dir():
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
    REPO_ROOT = REPO_DIR.resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next(
        (path for path in candidates if (path / "pyproject.toml").is_file() and (path / "src/vaaet").is_dir()),
        None,
    )
    if REPO_ROOT is None:
        raise RuntimeError("VAAET repository root not found")

os.chdir(REPO_ROOT)
install_command = [sys.executable, "-m", "pip", "install", "-q"]
if IN_COLAB:
    install_command.append(f"{REPO_ROOT}[training,visualization,database]")
else:
    install_command.extend(["-e", f"{REPO_ROOT}[training,visualization,database]"])
subprocess.check_call(install_command)

for module_name in tuple(sys.modules):
    if module_name == "vaaet" or module_name.startswith("vaaet."):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()

import vaaet

def validate_vaaet_origin(package: object, repo_root: Path, in_colab: bool) -> Path:
    package_file = getattr(package, "__file__", None)
    if not package_file:
        package_path = list(getattr(package, "__path__", ()))
        raise ImportError(
            "The 'vaaet' import resolved to a namespace package instead of the installed package. "
            f"Resolved locations: {package_path}. Re-run this setup cell."
        )
    origin = Path(package_file).resolve()
    expected_editable_root = (repo_root / "src/vaaet").resolve()
    if in_colab and repo_root.resolve() in origin.parents:
        raise ImportError(f"Colab must load the installed wheel, not repository path: {origin}")
    if not in_colab and origin.parent != expected_editable_root:
        raise ImportError(f"Local editable install has unexpected origin: {origin}")
    return origin

VAAET_PACKAGE_FILE = validate_vaaet_origin(vaaet, REPO_ROOT, IN_COLAB)
pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
    check=False,
)
pip_check_output = "\n".join(
    part.strip() for part in (pip_check.stdout, pip_check.stderr) if part.strip()
)
if pip_check.returncode == 0:
    print("✅ pip check: no broken requirements found")
else:
    print("⚠️ pip check detected conflicts in the managed notebook runtime:")
    print(pip_check_output or "No diagnostic output was returned")
    print("ℹ️ Continuing because workflow imports are validated explicitly below.")

def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not required"

print({name: package_version(name) for name in ("numpy", "tensorflow", "opencv-python-headless", "ultralytics-opencv-headless")})

import os
import shutil
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
import seaborn as sns
import sqlalchemy
import tensorflow as tf
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout, Input
from tensorflow.keras.models import Sequential

from vaaet.artifacts import MANIFEST_FILE, create_manifest
from vaaet.data.database import get_db_config, get_optional_db_config, hydrate_db_environment_from_colab, load_from_backup, load_telemetry
from vaaet.data.datasets import group_aware_train_test_split
from vaaet.data.persistence import persist_classified_telemetry
from vaaet.evaluation.reporting import build_class_support_notes, summarize_data_origin, summarize_resampled_balance, summarize_state_balance
from vaaet.features.engineering import engineer_features
from vaaet.features.labeling import assign_traffic_state
from vaaet.features.synthetic import augment_with_synthetic
from vaaet.inference.traffic_state import classify_telemetry_dataframe
from vaaet.logging import configure_logging
from vaaet.settings import DATA_PROCESSED_DIR, DATA_RAW_DIR, DB_ENV_VARS, DRIVE_ARTIFACT_DIR, FEATURE_COLS, LABELING_THRESHOLDS, MODEL_DIR, MODEL_VERSION, RANDOM_SEED, STATE_LABELS

hydrate_db_environment_from_colab()
configure_logging()
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | TensorFlow {tf.__version__} | GPU {bool(tf.config.list_physical_devices('GPU'))}")
print(f"Package: {VAAET_PACKAGE_FILE}")
print(f"✅ training workflow ready | root={REPO_ROOT} | commit={subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()}")

_MODEL_DIR = REPO_ROOT / MODEL_DIR
_DATA_DIR = REPO_ROOT / DATA_PROCESSED_DIR
_RAW_DIR = REPO_ROOT / DATA_RAW_DIR
for directory in (_MODEL_DIR, _DATA_DIR, _RAW_DIR):
    directory.mkdir(parents=True, exist_ok=True)


## Data Source — Raw Telemetry Acquisition

Data comes from the `traffic_data` table in PostgreSQL (AWS RDS), produced by the on-demand data-collection workflow. Each record represents one minute of processed video with 9 fields: average speed, counts by vehicle type (car, truck, bus, motorcycle, bicycle), and total.

The real dataset contains ~2 000 records from the backup `traffic_data.backup` (`pg_dump` format). The data loading cell uses a **three-tier fallback**:

1. **Tier 1** — Direct connection to PostgreSQL (RDS or local restore)
2. **Tier 2** — CSV cache at `data/raw/traffic_data_raw.csv` (saved on first successful load)
3. **Tier 3** — Binary `pg_dump` backup at `data/raw/traffic_data.backup` → converted via `pg_restore`

If running on **Colab without DB access**, upload the backup file using the cell below. The pipeline will auto-detect it and process it via `pg_restore`.

After loading, **Cell 2b** appends synthetic Accident and Congestion sequences (200 records total) because the real bridge data from Apr–Jul 2025 only contains Normal and Reduced traffic states. See `vaaet.features.synthetic` for generation details.

Credentials (if using Tier 1) are obtained via environment variables (`DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER`, `DB_PASSWORD`) or interactive input. They are never hardcoded or printed in outputs.

In [ ]:
# Cell 1b — Data Upload (Colab only)
#
# On Colab, if no CSV cache exists, upload one of:
#   - traffic_data.backup  (pg_dump binary → processed via pg_restore)
#   - traffic_data_raw.csv (direct CSV → used by Tier 2)
# On local, the files are expected at data/raw/.

_backup_dest = os.path.join(_RAW_DIR, "traffic_data.backup")
_csv_dest = os.path.join(_RAW_DIR, "traffic_data_raw.csv")

if IN_COLAB and not os.path.exists(_csv_dest):
    if not os.path.exists(_backup_dest):
        from google.colab import files  # type: ignore[import-untyped]
        print("📤 Upload traffic_data.backup (.backup) or traffic_data_raw.csv (.csv):")
        uploaded = files.upload()
        if uploaded:
            fname = list(uploaded.keys())[0]
            import shutil as _shutil
            if fname.endswith(".csv"):
                _shutil.move(fname, _csv_dest)
                print(f"✅ CSV saved to {_csv_dest} — will be loaded by Tier 2")
            else:
                _shutil.move(fname, _backup_dest)
                print(f"✅ Backup saved to {_backup_dest} — will be processed by Tier 3")
        else:
            print("⚠️ No file uploaded — Tier 2/3 fallback will be skipped")
    else:
        print(f"📂 Backup already exists: {_backup_dest}")
else:
    if os.path.exists(_csv_dest):
        print(f"📂 CSV cache available: {os.path.abspath(_csv_dest)}")
    elif os.path.exists(_backup_dest):
        print(f"📂 Backup available: {os.path.abspath(_backup_dest)}")
    else:
        print("📂 No data files present — will need DB connection (Tier 1)")

# A binary pg_dump needs a compatible PostgreSQL client (OS dependency).
PG_RESTORE_PATH: str | None = shutil.which("pg_restore")

if os.path.exists(_backup_dest) and not os.path.exists(_csv_dest):
    if IN_COLAB:
        _pg17_binary = Path("/usr/lib/postgresql/17/bin/pg_restore")
        if not _pg17_binary.is_file():
            print("📦 Configuring the official PostgreSQL PGDG repository...")
            _apt_env = {**os.environ, "DEBIAN_FRONTEND": "noninteractive"}
            _pgdg_dir = Path("/usr/share/postgresql-common/pgdg")
            _pgdg_key = _pgdg_dir / "apt.postgresql.org.asc"
            _pgdg_source = Path("/etc/apt/sources.list.d/pgdg.sources")
            try:
                subprocess.check_call(["apt-get", "update", "-qq"], env=_apt_env)
                subprocess.check_call(
                    ["apt-get", "install", "-y", "-qq", "curl", "ca-certificates"],
                    env=_apt_env,
                )
                subprocess.check_call(["install", "-d", str(_pgdg_dir)])
                subprocess.check_call(
                    [
                        "curl", "--fail", "--silent", "--show-error", "--retry", "3",
                        "--output", str(_pgdg_key),
                        "https://www.postgresql.org/media/keys/ACCC4CF8.asc",
                    ]
                )
                _os_release = {}
                for _line in Path("/etc/os-release").read_text(encoding="utf-8").splitlines():
                    if "=" in _line:
                        _key, _value = _line.split("=", 1)
                        _os_release[_key] = _value.strip().strip(chr(34))
                _codename = _os_release.get("VERSION_CODENAME")
                if not _codename:
                    raise RuntimeError("Could not determine the Ubuntu release codename")
                _architecture = subprocess.check_output(
                    ["dpkg", "--print-architecture"], text=True
                ).strip()
                _pgdg_source.write_text(
                    "Types: deb\n"
                    "URIs: https://apt.postgresql.org/pub/repos/apt\n"
                    f"Suites: {_codename}-pgdg\n"
                    f"Architectures: {_architecture}\n"
                    "Components: main\n"
                    f"Signed-By: {_pgdg_key}\n",
                    encoding="utf-8",
                )
                subprocess.check_call(["apt-get", "update", "-qq"], env=_apt_env)
                subprocess.check_call(
                    ["apt-get", "install", "-y", "-qq", "postgresql-client-17"],
                    env=_apt_env,
                )
            except (OSError, subprocess.CalledProcessError) as exc:
                raise RuntimeError(
                    "Could not install PostgreSQL 17 from the official PGDG repository. "
                    "Retry after reconnecting or upload traffic_data_raw.csv instead."
                ) from exc

        if not _pg17_binary.is_file() or not os.access(_pg17_binary, os.X_OK):
            raise RuntimeError(
                f"PostgreSQL 17 installation did not provide an executable: {_pg17_binary}. "
                "Upload traffic_data_raw.csv instead."
            )
        PG_RESTORE_PATH = str(_pg17_binary)

    if PG_RESTORE_PATH is None:
        raise RuntimeError(
            "No pg_restore executable is available. Install PostgreSQL 17 or upload "
            "traffic_data_raw.csv instead."
        )
    _pg_restore_version = subprocess.check_output(
        [PG_RESTORE_PATH, "--version"], text=True
    ).strip()
    print(f"✅ Backup reader ready: {_pg_restore_version} | {PG_RESTORE_PATH}")

In [ ]:
# Cell 2 — DB Connection + Telemetry Extraction
#
# Three-tier fallback:
#   1. PostgreSQL (AWS RDS or local) via vaaet.data.database.load_telemetry()
#      → Skipped automatically when DB env vars are not set (no blocking input())
#   2. Local CSV cache (data/raw/traffic_data_raw.csv)
#   3. Binary pg_dump backup (data/raw/traffic_data.backup) via pg_restore
#
# On first successful load from any source, a CSV cache is saved.

RAW_CSV_PATH: str = os.path.join(_RAW_DIR, "traffic_data_raw.csv")
BACKUP_PATH: str = os.path.join(_RAW_DIR, "traffic_data.backup")

df_raw: pd.DataFrame | None = None
DATA_SOURCE = "unknown"
_load_errors: list[str] = []

# Tier 1 — Direct DB connection (only if env vars are configured)
_has_db_env = all(os.environ.get(v) for v in DB_ENV_VARS)

if _has_db_env:
    try:
        db_config = get_db_config(interactive=False)
        df_raw = load_telemetry(config=db_config)
        DATA_SOURCE = "postgresql"
        df_raw.to_csv(RAW_CSV_PATH, index=False)
        print(f"✅ [Tier 1] Telemetry loaded from DB: {df_raw.shape[0]} records")
        print(f"   Raw CSV cache saved → {os.path.abspath(RAW_CSV_PATH)}")
    except Exception as e:
        _load_errors.append(f"Tier 1 (PostgreSQL): {e}")
        print(f"⚠️ [Tier 1] DB connection failed: {e}")
else:
    print("ℹ️ [Tier 1] Skipped — DB env vars not set (DB_HOST, DB_NAME, DB_USER, DB_PASSWORD)")

# Tier 2 — CSV cache
if df_raw is None and os.path.exists(RAW_CSV_PATH):
    try:
        df_raw = pd.read_csv(RAW_CSV_PATH)
        DATA_SOURCE = "csv-cache"
        print(f"✅ [Tier 2] Raw CSV loaded: {df_raw.shape[0]} records")
    except Exception as e:
        _load_errors.append(f"Tier 2 (CSV): {e}")
        print(f"⚠️ [Tier 2] CSV load failed: {e}")

# Tier 3 — Binary backup via pg_restore
if df_raw is None and os.path.exists(BACKUP_PATH):
    try:
        print("🔄 [Tier 3] Restoring from binary backup via pg_restore...")
        df_raw = load_from_backup(
            BACKUP_PATH,
            cache_csv=RAW_CSV_PATH,
            pg_restore_path=PG_RESTORE_PATH,
        )
        DATA_SOURCE = "postgres-backup"
        print(f"✅ [Tier 3] Backup restored: {df_raw.shape[0]} records")
    except Exception as e:
        _load_errors.append(f"Tier 3 (backup): {e}")
        print(f"🔴 [Tier 3] Backup restore failed: {e}")

# Final check
if df_raw is None:
    _diagnostics = "\n".join(f"  - {message}" for message in _load_errors)
    raise RuntimeError(
        "No data source could be loaded. Provide one of:\n"
        "  1. DB credentials via env vars (DB_HOST, DB_NAME, DB_USER, DB_PASSWORD)\n"
        "  2. CSV file at data/raw/traffic_data_raw.csv\n"
        "  3. pg_dump backup at data/raw/traffic_data.backup\n"
        + (f"Diagnostics:\n{_diagnostics}" if _diagnostics else "No source was found.")
    )

print(f"\n📊 Dataset: {df_raw.shape[0]} records × {df_raw.shape[1]} columns")
print(f"   Time range: {df_raw['record_time'].min()} → {df_raw['record_time'].max()}")
display(df_raw.describe().round(2)) if "display" in dir() else print(df_raw.describe().round(2))

In [ ]:
# Cell 2b — Synthetic Data Augmentation
#
# The Belgrano Bridge dataset (Apr–Jul 2025) contains only Normal and
# Reduced traffic — no Congested or Accident events occurred.
# We inject physically plausible synthetic sequences so the classifier
# can learn all 4 states.  Synthetic IDs start at 50001 and timestamps
# fall before the real data range (2025-04-21…27).

if "df_raw" not in globals() or not isinstance(df_raw, pd.DataFrame) or df_raw.empty:
    raise RuntimeError(
        "Raw telemetry is unavailable. Run Cell 1b and then Cell 2, and ensure "
        "that Cell 2 finishes successfully before running synthetic augmentation."
    )

_n_before = len(df_raw)

df_raw = augment_with_synthetic(
    df_raw,
    n_accident_seq=10,
    n_congestion_seq=10,
    records_per_seq=10,
    seed=RANDOM_SEED,
)

_n_synthetic = len(df_raw) - _n_before

print(f"✅ Synthetic augmentation: {_n_synthetic} records added")
print(f"   Accident sequences: 10 × 10 = 100 records")
print(f"   Congestion sequences: 10 × 10 = 100 records")
print(f"   Total dataset: {len(df_raw)} records ({_n_before} real + {_n_synthetic} synthetic)")
print(f"   Synthetic IDs ≥ 50001 (distinguishable from real data)")

origin_summary = summarize_data_origin(df_raw)
print("\n📋 Dataset provenance:")
display(origin_summary) if "display" in dir() else print(origin_summary.to_string(index=False))


## Feature Engineering — From Raw Telemetry to 19 Features

Raw telemetry (speed + counts) does not capture relationships between consecutive records or temporal patterns. Feature engineering produces the 19 canonical variables the model can exploit:

| Feature | Origin | Domain Justification |
|---|---|---|
| `avg_speed` | Direct | Primary indicator of vehicular flow |
| `total_vehicles` | Direct | Absolute traffic volume |
| `count_car` ... `count_bicycle` | Direct (5) | Vehicle composition — trucks and buses impact flow differently than cars |
| `heavy_vehicle_ratio` | Derived | Heavy vehicle proportion — heavy traffic degrades flow more |
| `delta_speed` | Derived (diff) | Acceleration/deceleration between consecutive minutes |
| `delta_count` | Derived (diff) | Volume change rate — detects accumulation |
| `transition_flag` | Derived | Binary signal: simultaneous sharp changes in speed (>8 km/h) and volume (>3 vehicles) |
| `speed_variance` | Derived (rolling) | Recent variability — unstable vs stable traffic |
| `cumulative_delta_speed` | Derived | Accumulated speed trend within the sequence |
| `low_speed_persistence` | Derived | Duration of sustained low-speed conditions |
| `speed_measurement_quality` | Derived | Reliability of the underlying speed estimate |
| `near_zero_motion_ratio` | Direct/derived | Share of tracks with near-zero motion |
| `stationary_confirmed_ratio` | Direct/derived | Share of tracks confirmed as stationary |
| `hour_of_day` | Temporal | Circadian traffic patterns (rush hour, nighttime) |
| `weather_condition` | Simulated | Environmental condition proxy based on hour (nighttime=risk) |

Derived features (`delta_*`, `speed_variance`) introduce NaN in the first records, which are dropped.

> **Note**: The dataset now includes synthetic Accident and Congestion sequences (Cell 2b) appended after real records.  Feature engineering processes them identically.

In [ ]:
# Cell 3 — Feature Engineering
#
# Uses vaaet.features.engineering.engineer_features() and vaaet.settings.FEATURE_COLS
# (imported in Cell 1). No inline duplication.

df_features = engineer_features(df_raw)

# Save features CSV for reproducibility (NOT to be used as raw data fallback)
csv_path = os.path.join(_DATA_DIR, "traffic_telemetry.csv")
df_features.to_csv(csv_path, index=False)

print(f"✅ Features engineered: {df_features.shape[0]} records × {df_features.shape[1]} columns")
print(f"   Features CSV saved → {os.path.abspath(csv_path)}")
print(f"\n📊 Correlation with avg_speed:")
corr = df_features[FEATURE_COLS].corr()["avg_speed"].drop("avg_speed").sort_values()
print(corr.to_string())

## Auto-Labeling — Bridge-Calibrated Engineering Rules

Without manual annotation of thousands of records, we use engineering rules as a ground truth proxy. Thresholds are **calibrated to the Belgrano Bridge real data distribution** (P25 speed ≈ 7.78 km/h, median vehicles ≈ 3, P75 vehicles ≈ 6):

- **Accident (3)** — most severe: speed ~0 km/h after sudden braking (`delta_speed < -15`) detected within a recent 5-minute window, sustained ≥2 consecutive records at <2 km/h. Assigned first so it is not overwritten.
- **Congested (2)**: speed <7 km/h with density >8 veh/min, sustained ≥2 records.
- **Reduced (1)**: speed between 7–25 km/h with moderate density (5–12 veh/min).
- **Normal (0)**: everything else (default catch-all).

The real dataset only contained Normal and Reduced events.  Synthetic sequences (Cell 2b) provide Congested and Accident samples so the model learns all 4 states.

**Known limitation**: these labels are NOT human ground truth. A SISE operator corrects this via HITL (fields `is_human_validated` and `human_override_state` in the `traffic_classifications` table) in the the inference workflow feedback loop. See [bias and limitations](../../docs/ml/bias-and-limitations.md).

In [ ]:
# Cell 4 — Auto-Labeling + Class Distribution
#
# Uses vaaet.features.labeling.assign_traffic_state() and vaaet.settings.STATE_LABELS
# (imported in Cell 1). No inline duplication.

df_features["traffic_state"] = assign_traffic_state(df_features)

# Distribution
dist = df_features["traffic_state"].value_counts().sort_index()
print("📊 Traffic state distribution:")
for code, count in dist.items():
    pct = 100 * count / len(df_features)
    print(f"   {STATE_LABELS[code]:>10} ({code}): {count:>5} records ({pct:.1f}%)")

# Verify at least 2 classes exist
n_classes = dist.index.nunique()
if n_classes < 2:
    print("🔴 Only 1 class found. Thresholds do not discriminate in this dataset.")
else:
    print(f"\n✅ {n_classes} classes detected")

# Classes without samples
for code, label in STATE_LABELS.items():
    if code not in dist.index:
        print(f"⚠️  Class '{label}' ({code}) has no samples — will be excluded from training")

# Visualization
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#2ecc71", "#f39c12", "#e74c3c", "#8e44ad"]
bars = ax.bar(
    [STATE_LABELS[c] for c in sorted(dist.index)],
    [dist[c] for c in sorted(dist.index)],
    color=[colors[c] for c in sorted(dist.index)],
)
ax.set_ylabel("Records")
ax.set_title("Traffic State Distribution (Auto-Labeling)")
for bar, count in zip(bars, [dist[c] for c in sorted(dist.index)]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(count), ha="center", fontsize=10)
plt.tight_layout()
plt.show()

support_summary = summarize_state_balance(df_features)
print("\n📋 Support by origin:")
display(support_summary) if "display" in dir() else print(support_summary.to_string(index=False))

print("\n📝 Support notes:")
for note in build_class_support_notes(df_features):
    print(f"   - {note}")


## Balancing and Partitioning — SMOTE + Stratification

The real dataset is strongly imbalanced: ~80% Normal expected, with Accident potentially <1%. Training a model directly would produce a classifier that ignores minority classes.

**Strategy**:
1. **StandardScaler**: Normalizes features to mean=0, std=1 (required for neural networks)
2. **Train/Test split** (80/20): Stratified to maintain original proportions in both sets
3. **SMOTE** (Synthetic Minority Over-sampling Technique): Applied **only to the training set** to generate synthetic samples from minority classes. The test set remains intact as a realistic evaluation

The scaler is exported as an artifact (`feature_scaler.joblib`) so that production inference uses the same transformation.

In [ ]:
# Cell 5 — Group-Aware Split + SMOTE

split = group_aware_train_test_split(
    df_features,
    target_col="traffic_state",
    group_col="clip_id",
    time_col="record_time",
    fallback_window="15min",
    test_size=0.2,
    random_state=RANDOM_SEED,
)

train_frame = df_features.iloc[split.train_idx].copy()
test_frame = df_features.iloc[split.test_idx].copy()

X_train_raw = train_frame[FEATURE_COLS].values
X_test_raw = test_frame[FEATURE_COLS].values
y_train = train_frame["traffic_state"].values
y_test = test_frame["traffic_state"].values

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"📊 Leakage-aware partition:")
print(f"   Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"   Train groups: {split.groups.iloc[split.train_idx].nunique()} | Test groups: {split.groups.iloc[split.test_idx].nunique()}")
print(f"   Train distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"   Test distribution: {dict(zip(*np.unique(y_test, return_counts=True)))}")

# SMOTE (training only)
train_counts = np.bincount(y_train)
min_class_count = train_counts[train_counts > 0].min()
k_neighbors = min(5, min_class_count - 1) if min_class_count > 1 else 1

if min_class_count < 2:
    print("⚠️ Class with <2 samples in train. SMOTE disabled — proceeding without balancing.")
    X_train_res, y_train_res = X_train, y_train
else:
    sm = SMOTE(random_state=RANDOM_SEED, k_neighbors=k_neighbors)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    print(f"\n✅ SMOTE applied (k_neighbors={k_neighbors}):")
    print(f"   Balanced train: {X_train_res.shape[0]} samples")
    print(f"   Distribution: {dict(zip(*np.unique(y_train_res, return_counts=True)))}")

# Export scaler
scaler_path = os.path.join(_MODEL_DIR, "feature_scaler.joblib")
joblib.dump(scaler, scaler_path)
print(f"\n💾 Scaler saved → {os.path.abspath(scaler_path)}")

balance_after_smote = summarize_resampled_balance(y_train, y_train_res)
print("\n📋 Train balance before/after resampling:")
display(balance_after_smote) if "display" in dir() else print(balance_after_smote.to_string(index=False))


## Model Architecture — Tabular MLP

The model is a **Multi-Layer Perceptron (MLP)** implemented with `tf.keras.Sequential`. The architecture is deliberately simple — designed to validate the complete pipeline. A future iteration may evolve to LSTM with temporal memory.

**Why these dimensions?**
- **Dense(64)**: Input layer with sufficient capacity to learn non-linear combinations of 19 features
- **Dense(32)**: Compression layer that forces more abstract representations
- **BatchNormalization**: Stabilizes and accelerates training by normalizing activations between layers
- **Dropout(0.3 → 0.2)**: Decreasing regularization — more aggressive near the input (where there is more redundancy)
- **Softmax(n_classes)**: Probability distribution over the 4 states

In [ ]:
# Cell 6 — Model Definition + Training

# Dynamic number of classes based on the resampled TRAIN labels.
# We encode original traffic-state codes into contiguous indices (0..n-1)
# so sparse_categorical_crossentropy remains valid when some classes are absent.
train_classes = np.array(sorted(np.unique(y_train_res)), dtype=int)
class_to_index = {cls: idx for idx, cls in enumerate(train_classes)}
index_to_class = {idx: cls for cls, idx in class_to_index.items()}

y_train_res_encoded = np.array([class_to_index[c] for c in y_train_res], dtype=np.int32)

n_classes: int = len(train_classes)
n_features: int = X_train_res.shape[1]

print(f"🏗️ Building MLP model: {n_features} features → {n_classes} classes")
print(f"   Original class codes (train): {train_classes.tolist()}")
print(f"   Encoded class indices: {list(range(n_classes))}")

# Architecture
model = Sequential([
    Input(shape=(n_features,)),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation="relu"),
    BatchNormalization(),
    Dropout(0.2),
    Dense(n_classes, activation="softmax"),
], name="traffic_state_classifier")

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

# Callbacks
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        patience=5,
        factor=0.5,
        min_lr=1e-6,
        verbose=1,
    ),
]

# Training
print("\n🚀 Starting training...")
history = model.fit(
    X_train_res,
    y_train_res_encoded,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1,
)

# Training visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history.history["loss"], label="Train Loss")
ax1.plot(history.history["val_loss"], label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss During Training")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(history.history["accuracy"], label="Train Accuracy")
ax2.plot(history.history["val_accuracy"], label="Val Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy During Training")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_epoch = np.argmin(history.history["val_loss"]) + 1
print(f"\n✅ Training completed — best epoch: {best_epoch}")

## Evaluation — Classification Metrics

The key metrics for this classifier are:

- **F1-macro** ≥ 0.85: Unweighted average of per-class F1. Equally penalizes performance on rare classes (Accident) and frequent classes (Normal)
- **Per-class Recall** > 0: Especially for Accident — a recall of 0 would mean the model never detects this critical class
- **Confusion matrix**: Identifies systematic confusions (e.g., Normal↔Reduced is the most likely confusion pair)

The model is exported as `.keras` (native standalone format) along with the label mapping.

In [ ]:
# Cell 7 — Evaluation + Model Export

# Predict on test set (model outputs encoded class indices)
y_proba = model.predict(X_test)
y_pred_encoded = y_proba.argmax(axis=1)

# Decode predictions back to original traffic-state codes
y_pred = np.array([index_to_class[idx] for idx in y_pred_encoded], dtype=int)

# Present class names
present_classes = sorted(np.unique(np.concatenate([y_test, y_pred])))
target_names = [STATE_LABELS[c] for c in present_classes]

# Classification Report
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
report = classification_report(
    y_test, y_pred,
    labels=present_classes,
    target_names=target_names,
    zero_division=0,
)
print(report)

# F1-macro
f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
print(f"{'F1-macro':>15}: {f1_macro:.4f}")

if f1_macro >= 0.85:
    print(f"✅ F1-macro MEETS the target (≥ 0.85)")
else:
    print(f"⚠️  F1-macro BELOW target (≥ 0.85) — review balancing or thresholds")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=present_classes)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names,
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — F1-macro: {f1_macro:.4f}")
plt.tight_layout()
plt.show()

# Per-class recall
print("\n📊 Per-class recall:")
for i, cls in enumerate(present_classes):
    row_sum = cm[i].sum()
    recall = cm[i, i] / row_sum if row_sum > 0 else 0.0
    status = "✅" if recall > 0 else "🔴"
    print(f"   {status} {STATE_LABELS[cls]:>10}: {recall:.4f}")

# Export model
model_path = os.path.join(_MODEL_DIR, "traffic_classifier.keras")
model.save(model_path)

# Export the canonical mapping required by the portable serving contract.
label_mapping = dict(STATE_LABELS)
label_path = os.path.join(_MODEL_DIR, "label_mapping.joblib")
joblib.dump(label_mapping, label_path)
create_manifest(
    _MODEL_DIR,
    metrics={"f1_macro": float(f1_macro)},
    data_provenance={
        "origin": "training-notebook",
        "dataset_source": str(DATA_SOURCE),
        "record_count": int(len(df_features)),
        "real_record_count_before_engineering": int(_n_before),
        "synthetic_record_count_before_engineering": int(_n_synthetic),
        "synthetic_data_included": bool(_n_synthetic > 0),
    },
)

print(f"\n💾 Artifacts exported:")
print(f"   Model  → {os.path.abspath(model_path)} ({os.path.getsize(model_path) / 1024:.1f} KB)")
print(f"   Labels → {os.path.abspath(label_path)} (classes: {list(label_mapping.values())})")
print(f"   Scaler → {os.path.abspath(os.path.join(_MODEL_DIR, 'feature_scaler.joblib'))}")

print("\n📝 Support notes:")
for note in build_class_support_notes(df_features):
    print(f"   - {note}")

## K-Fold Cross-Validation (Optional)

Provides a more robust estimate of model generalization by training and evaluating on 5 different data splits. This addresses potential optimistic bias from a single 80/20 split.

- **StratifiedKFold**: Maintains class proportions in each fold
- **SMOTE per fold**: Applied only to training folds (no data leakage)
- Reports mean +/- std of F1-macro across all folds

In [ ]:
# Cell 7b — K-Fold Cross-Validation (Optional)
#
# Run this cell for a more robust generalization estimate.
# It does NOT replace the single-split model (exported in Cell 7).

from sklearn.model_selection import StratifiedKFold

N_FOLDS: int = 5
kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

# Use the ORIGINAL (unscaled) feature matrix
X_all = df_features[FEATURE_COLS].values
y_all = df_features["traffic_state"].values

fold_f1_scores: list[float] = []

print(f"🔁 {N_FOLDS}-Fold Stratified Cross-Validation")
print("=" * 50)

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_all, y_all), 1):
    X_tr, X_val = X_all[train_idx], X_all[val_idx]
    y_tr, y_val = y_all[train_idx], y_all[val_idx]

    # Scale per fold
    fold_scaler = StandardScaler()
    X_tr = fold_scaler.fit_transform(X_tr)
    X_val = fold_scaler.transform(X_val)

    # SMOTE per fold (training only)
    tr_counts = np.bincount(y_tr)
    min_cls = tr_counts[tr_counts > 0].min()
    k_nn = min(5, min_cls - 1) if min_cls > 1 else 1

    if min_cls >= 2:
        sm_fold = SMOTE(random_state=RANDOM_SEED, k_neighbors=k_nn)
        X_tr, y_tr = sm_fold.fit_resample(X_tr, y_tr)

    # Fold-local class encoding keeps sparse labels in range [0, n_classes_fold-1]
    fold_classes = np.array(sorted(np.unique(y_tr)), dtype=int)
    fold_class_to_index = {cls: idx for idx, cls in enumerate(fold_classes)}
    fold_index_to_class = {idx: cls for cls, idx in fold_class_to_index.items()}
    y_tr_encoded = np.array([fold_class_to_index[c] for c in y_tr], dtype=np.int32)

    # Build model (identical architecture)
    fold_model = Sequential([
        Input(shape=(X_tr.shape[1],)),
        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation="relu"),
        BatchNormalization(),
        Dropout(0.2),
        Dense(len(fold_classes), activation="softmax"),
    ])
    fold_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    fold_model.fit(
        X_tr, y_tr_encoded,
        epochs=200,
        batch_size=32,
        validation_split=0.2,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=0),
        ],
        verbose=0,
    )

    y_pred_fold_encoded = fold_model.predict(X_val, verbose=0).argmax(axis=1)
    y_pred_fold = np.array([fold_index_to_class[idx] for idx in y_pred_fold_encoded], dtype=int)

    fold_f1 = f1_score(y_val, y_pred_fold, average="macro", zero_division=0)
    fold_f1_scores.append(fold_f1)
    print(f"   Fold {fold_idx}: F1-macro = {fold_f1:.4f}")

mean_f1 = np.mean(fold_f1_scores)
std_f1 = np.std(fold_f1_scores)

print("=" * 50)
print(f"📊 K-Fold F1-macro: {mean_f1:.4f} ± {std_f1:.4f}")

if mean_f1 >= 0.85:
    print(f"✅ Cross-validated F1-macro MEETS target (≥ 0.85)")
else:
    print(f"⚠️  Cross-validated F1-macro BELOW target (≥ 0.85)")
    print(f"   Consider adjusting labeling thresholds or model architecture")

In [ ]:
# Cell 7c — Export Artifacts to Google Drive (Colab only)
#
# Copies trained artifacts to Google Drive so the inference workflow can load them
# even after a Colab runtime reset.  Skipped silently when running locally.

import shutil

if IN_COLAB:
    try:
        from google.colab import drive  # type: ignore[import-untyped]
        drive.mount("/content/drive", force_remount=False)

        _drive_dest = os.path.join("/content/drive", DRIVE_ARTIFACT_DIR)
        os.makedirs(_drive_dest, exist_ok=True)

        _artifact_files = [
            os.path.join(_MODEL_DIR, "traffic_classifier.keras"),
            os.path.join(_MODEL_DIR, "feature_scaler.joblib"),
            os.path.join(_MODEL_DIR, "label_mapping.joblib"),
            os.path.join(_MODEL_DIR, MANIFEST_FILE),
        ]

        _copied = 0
        for src_path in _artifact_files:
            if os.path.isfile(src_path):
                shutil.copy2(src_path, _drive_dest)
                _copied += 1
            else:
                print(f"⚠️  Not found (skipped): {src_path}")

        if _copied == len(_artifact_files):
            print(f"✅ {_copied} artifacts exported to Google Drive:")
            print(f"   {_drive_dest}")
        else:
            print(f"⚠️  Only {_copied}/{len(_artifact_files)} artifacts copied")
    except Exception as e:
        print(f"⚠️  Drive export skipped: {e}")
        print("   Artifacts are available locally — run inference in the same session")
else:
    print("ℹ️  Local environment — Drive export skipped")
    print(f"   Artifacts at: {os.path.abspath(_MODEL_DIR)}")

## Persistence — Two New Tables with FK

The classifier results are persisted in PostgreSQL with full traceability:

```
traffic_data (raw acquisition)
    ↓ FK: source_record_id
telemetry_raw (19 engineered features)
    ↓ FK: telemetry_id
traffic_classifications (prediction + HITL)
```

- **`telemetry_raw`**: Stores the 19 calculated features, with FK to the original record in `traffic_data`. Enables training reproducibility and auditing which data fed each prediction.
- **`traffic_classifications`**: Stores the model prediction, confidence, model version, and HITL fields (`is_human_validated`, `human_override_state`, `validated_at`) that will be activated when a SISE operator can confirm/dismiss alerts.

Persistence is **optional**: if no DB is configured, the notebook works completely and exports artifacts locally.

In [ ]:
# Cell 8 — Persist Results (Optional)
#
# Heavy persistence logic lives in vaaet.data.persistence. The notebook only
# orchestrates the call and degrades cleanly when no DB is configured.

try:
    db_config = get_optional_db_config(interactive=False)
    if db_config is None:
        print("⚠️ No DB configuration — results remain local only")
        print(f"   Artifacts: {os.path.abspath(_MODEL_DIR)}")
    else:
        df_persist = classify_telemetry_dataframe(
            df_features,
            model,
            scaler,
            label_mapping=label_mapping,
            model_version=MODEL_VERSION,
        )
        persisted = persist_classified_telemetry(
            df_persist,
            config=db_config,
            model_version=MODEL_VERSION,
        )
        print(f"✅ Persistence completed: {persisted.telemetry_rows} telemetry rows | {persisted.classification_rows} classifications")
except NameError:
    print("⚠️ Model artifacts not ready — run Cells 5-7 first")
except Exception as e:
    print(f"🔴 Persistence error: {e}")
    print("   Model and artifacts remain available locally")
